# <font color="purple"><b>How to Win a Datathon 101</b></font>

### Dataverse 2026 Workshop
**Part 1:** Meet & clean your data · **Part 2:** Exploratory data analysis (EDA) · **Part 3:** Data modelling

**The (hypothetical) client:** the City of Vancouver wants to run a campaign that gets residents and visitors exploring more neighbourhoods, driving foot traffic to local businesses. They want to know **which areas to promote, and when**.

**The data:** the City of Vancouver's *Public trees* dataset (City of Vancouver Open Data Portal, Open Government Licence – Vancouver).

https://opendata.vancouver.ca/explore/assets/public-trees/export/

---

**How to use this notebook**
1. Set up your folders: create a folder called `data` with two subfolders inside it, `raw` and `processed`. Put `public-trees.csv` inside `data/raw` (in Colab: click the File icon on the left → right-click → **New folder**, then upload the file into `data/raw`).
2. Run each cell in order with **Shift + Enter**.
3. **TRY IT YOURSELF!** cells are where you get to practice. They come in three levels: Easy · Medium · Hard.
4. Stuck? Try the hint first, then click **Show Solution**. There's often more than one right answer!

In [ ]:
import pandas as pd

# Show more columns when we print a table
pd.set_option("display.max_columns", 20)

---

# Part 1: Meet & Clean Your Data

Before we analyze anything, we need to **meet our data**. Just as you'd get to know a real client, you want to get to know your data: what's in it, what's missing, and what looks off.

In real life (jobs, research *and* datathons), getting to know and cleaning your data often takes up the **biggest chunk of your time**. That's completely normal, and it's a great investment: messy data leads to wrong answers, no matter how fancy your model is.

Every dataset is different, so there's no perfect checklist. But here are six things worth checking first:

| Check | Question to ask |
|---|---|
| **1. Structure** | What does one row represent? How large is the dataset? |
| **2. Relevant variables** | Which columns help answer our question? |
| **3. Missing values** | Are there any missing values, and if so, how much? *(Tip: look out for values like `0`, `"N/A"`, etc.)* |
| **4. Data types** | Are numbers, dates and categories stored correctly? |
| **5. Inconsistencies** | Are names, formats, and units consistent? |
| **6. Impossible values & outliers** | Are there any values that don't make sense? |

**TIP:** Your first look at the data guides your cleaning decisions. There's rarely one "right" answer, so the key is to make sensible choices and **write down why**.

## 0. Load the data
NOTE: This file uses **semicolons (`;`)**, not commas, to separate columns. If you forget `sep=";"`, everything ends up squished into one column.

In [ ]:
trees = pd.read_csv(..., sep=...)
trees_raw = trees.copy()   # keep an untouched copy, just in case!
trees.head()  # shows the first 5 rows of your dataframe

---

## 1. Structure
**What does one row represent? How large is the dataset?**

In [ ]:
trees.shape   # (rows, columns)

It looks like one row = one tree... but let's check. What are the most common "tree" names?

In [ ]:
trees["Common name"].value_counts().head(15)

**TRY IT YOURSELF! (Easy):** Return the single most common tree in Vancouver with its count.

*Hint: `value_counts()` sorts from most to least common. How do you keep just the first row?*

In [ ]:
# your code here

<details>
<summary>Show solution</summary>

```python
trees["Common name"].value_counts().head(1)
```

The **Kwanzan flowering cherry** is the most common tree in Vancouver.

</details>

**TRY IT YOURSELF! (Medium):** How many *different kinds* of trees (common names) are there?

*Hint: Look up the pandas method `nunique()`.*

In [ ]:
# your code here

<details>
<summary>Show solution</summary>

```python
trees["Common name"].nunique()
```

Over 1,300! That's a lot of categories, which is something to keep in mind before making a chart with all of them.

</details>

---

## 2. Relevant variables
**Which columns help answer our question?**

In [ ]:
trees.columns

**Decision:** we keep the columns about **what** the tree is, **how big** it is, and **where** it is. We drop `Geom` because it repeats the location in a messier format.

We'll also rename the columns to short, lowercase names with no spaces, which makes them much easier to type!

In [ ]:
trees = trees.rename(columns={
    "Asset ID": "...",
    "Address": "address",
    "Common name": "common_name",
    "Genus name": "...",
    "Species name": "species",
    "Cultivar name": "cultivar",
    "height_(m)": "height_m",
    "diameter_(cm)": "...",
    "Local Area": "neighbourhood",
    "geo_point_2d": "location",
})

trees = trees.drop(columns=["Geom"])
trees.head()

**TRY IT YOURSELF (Medium):** Return the number of columns we have now.

In [ ]:
# your code here


<details>
<summary>Show solution</summary>

```python
trees.shape[1]
```

We went from 11 to 10 columns: `Geom` was dropped (and the rest were renamed).

</details>

---

## 3. Missing values
**Are there any missing values, and if so, how many?**

In [ ]:
trees.isna().sum()

Thousands of trees have no `neighbourhood`. Before deleting them, let's **look at where they are**.

In [ ]:
trees.loc[trees["neighbourhood"].isna(), "address"].value_counts().head(5)

**2000 W Georgia St is the address of Stanley Park's entrance!** Stanley Park isn't one of the City's official neighbourhoods, so its trees have NA in its place.

**Decision:** if we had just deleted missing rows, we'd lose one of Vancouver's most famous parks. For simplicity and time, we'll label those trees "Stanley Park" and mark the rest as "Unknown".

In [ ]:
is_stanley = trees[...] == "2000 W GEORGIA ST"
trees.loc[is_stanley, ...] = "Stanley Park"
trees["neighbourhood"] = trees["neighbourhood"].fillna(...)

(trees["neighbourhood"] == "Stanley Park").sum()

### Sneaky (hidden) missing values
Pandas only counts real blanks as missing. But missing data can hide behind values like `"NONE"`, `"XX"`, `0` or `"Unknown"`.

In [ ]:
print("Cultivar = 'NONE':", (trees["cultivar"] == "NONE").sum())
print("Species  = 'XX':  ", (trees["species"] == "XX").sum())

**Decision:** we turn these into real missing values (`NaN`), so pandas treats them correctly. We also drop the few trees with no name at all, since we can't tell what they are.

In [ ]:
trees["cultivar"] = trees["cultivar"].replace("___", pd.NA)
trees["species"] = trees["species"].replace("___", pd.NA)

trees = trees.___(subset=["___"])

trees.___().sum()

**TRY IT YOURSELF (Easy):** Which column has the most missing values now?

*Hint: use `.sort_values(ascending=False)` as part of your solution.*

In [ ]:
# your code here

<details>
<summary>Show solution</summary>

```python
trees.isna().sum().sort_values(ascending=False)
```

**`cultivar`**, but that's fine! The cultivar is a very specific variety name, and we don't need it to answer our client's question. Not every missing value needs fixing.

</details>

**TRY IT YOURSELF (Medium):** For trees with an "Unknown" neighbourhood, find the 10 most common addresses and how many trees are at each one.

In [ ]:
# your code here

<details>
<summary>Show solution</summary>

```python
trees.loc[trees["neighbourhood"] == "Unknown", "address"].value_counts().head(10)
```
</details>

---

## 4. Data types
<font color="red"><b>Additional exercises:</b> we won't have time for these during the workshop. Try them on your own afterwards; hints and solutions are included!</font>

**Are numbers, dates and categories stored correctly?**

In [ ]:
trees.dtypes

Height and diameter are numbers. But `location` is stored as **text**, like `"49.2556, -123.1758"`, so we can't put it on a map yet.

**Decision:** split it into two numeric columns, `lat` and `lon`.

In [ ]:
trees = trees.dropna(subset=["location"])   # one tree has no location

lat_lon = trees["location"].str.split(",", expand=True)
trees["lat"] = lat_lon[0].astype(float)
trees["lon"] = lat_lon[1].astype(float)

trees = trees.drop(columns=["location"])
trees[["common_name", "lat", "lon"]].head()

**TRY IT YOURSELF (Medium):** Check the smallest and largest `lat` and `lon` values. Do they look like Vancouver? (Vancouver is at roughly 49.2°N, 123.1°W.)

*Hint: Try `.describe()` on just those two columns.*

In [ ]:
# your code here

<details>
<summary> Show solution</summary>

```python
trees[["lat", "lon"]].describe()
```

Latitudes are all around 49.2–49.3 and longitudes around −123.0 to −123.2, so everything is in Vancouver. Sanity-checking ranges like this is a quick way to catch errors (e.g., flipped lat/lon).

</details>

---

## 5. Inconsistencies

<font color="red"><b>Additional exercises:</b> we won't have time for these during the workshop. Try them on your own afterwards; hints and solutions are included!</font>

**Is the same thing written in different ways? Are there duplicates?**

Let's first address an example of a typo: is anything spelled "COMMOM" instead of "COMMON"?

In [ ]:
trees.loc[trees["common_name"].str.contains("COMMOM"), "common_name"].value_counts()

In [ ]:
# Fix the typo, and squash any double spaces into single spaces
trees["common_name"] = (trees["common_name"]
                        .str.replace("COMMOM", "COMMON")
                        .str.replace(r"\s+", " ", regex=True))

### Duplicates
Each tree has a unique ID, so `duplicated()` alone finds nothing. But are some trees listed twice with **different IDs**? Let's compare every column *except* the ID.

In [ ]:
other_cols = trees.columns.drop("tree_id")
dupes = trees.duplicated(subset=other_cols)
print("Duplicate trees:", dupes.sum())

# Example: same tree, same exact spot, two different IDs
trees[trees.duplicated(subset=other_cols, keep=False)].sort_values(["lat", "lon"]).head(4)

In [ ]:
trees = trees[~dupes]
trees.shape

**TRY IT YOURSELF (Hard):** List the common names that appear only **once**. Can you spot any that look like mistakes (i.e., typos vs. one-offs)?

In [ ]:
# your code here

<details>
<summary>Show solution</summary>

```python
counts = trees["common_name"].value_counts()
counts[counts == 1]
```

Most are real (just rare) trees, but this is a quick way to find candidates to double-check. Real datasets almost always have a few more typos hiding somewhere!

</details>

---

## 6. Impossible values & outliers

<font color="red"><b>Additional exercises:</b> we won't have time for these during the workshop. Try them on your own afterwards; hints and solutions are included!</font>


**Are there any values that don't make sense?**

In [ ]:
trees[["height_m", "diameter_cm"]].describe()

A tree with a diameter of **0 cm**? And look at that max: let's see the biggest "trees".

In [ ]:
print("Diameter = 0:", (trees["diameter_cm"] == 0).sum())
print("Height = 0:  ", (trees["height_m"] == 0).sum())

trees.nlargest(5, "diameter_cm")[["common_name", "height_m", "diameter_cm", "neighbourhood"]]

A pin oak that's over **66 metres wide**? That's wider than most buildings. Definitely a data entry error!

**Decision:** keep trees with a height above 0 and a diameter between 0 and 500 cm. There's no single "right" cut-off. What matters is that you choose one sensibly and **explain why**.

In [ ]:
valid = (trees["height_m"] > 0) & (trees["diameter_cm"] > 0) & (trees["diameter_cm"] < 500)
print("Removing:", (~valid).sum(), "trees")

trees = trees[valid]
trees[["height_m", "diameter_cm"]].describe()

**TRY IT YOURSELF (Medium):** We only filtered *diameter* outliers. What are the 5 **tallest** trees left?

*Hint: Use `nlargest()` in your solution.*

In [ ]:
# your code here

<details>
<summary>Show solution</summary>

```python
trees.nlargest(5, "height_m")[["common_name", "height_m", "diameter_cm", "neighbourhood"]]
```

Three of the five tallest trees are in Kensington-Cedar Cottage, and some of them are species that are usually quite small, like the pyramidal hornbeam. Other trees like the Douglas fir are native BC trees known for growing very tall.

</details>

---

## 7. Save our clean data

We'll save our cleaned data in a new subfolder called `processed`, inside our `data` folder.

**Why keep raw and processed data separate?**
- **Your original data stays untouched,** so you can always start over if something goes wrong.
- **Your team knows which file to use,** so everyone works from the same clean version.
- **Your project stays organized** as you create more files.

In [ ]:
trees.to_csv("data/processed/trees_clean.csv", index=False)
print(f"Started with {len(trees_raw):,} rows, now {len(trees):,} clean trees.")

---

# Part 2: Exploratory Data Analysis (EDA)

Now that our data is clean, it's time to get curious! EDA sounds intimidating, but it's really just **asking questions about your data and answering them with simple plots and summaries**.

There's no single right way to do EDA, so treat these as pointers, not rules:

| Pointer | What it means |
|---|---|
| **1. Start with a question** | What does your client want to know? Let that guide what you look at. |
| **2. One variable at a time** | What's common? What's rare? What's the range? |
| **3. Compare & connect** | How do groups differ? Do two things move together? |
| **4. Follow the surprises** | What looks intriguing, and why? Surprises are clues! |

**Keep it simple: a few clear plots beat a dozen confusing ones.**

### From questions to charts
**Answering the questions in our four pointers also helps you decide which chart to use.** The kind of question you're asking points you to the right chart:

| Question | Chart | Example from our data | Tip |
|---|---|---|---|
| What's common or rare? | **Bar chart** | Top 10 trees | Sort the bars; go horizontal for long labels |
| How are values spread out? | **Histogram** | Tree heights | Try a few bin sizes, since patterns can hide |
| How do groups compare? | **Box plot** or bar chart of medians | Cherry size by neighbourhood | Medians hold up better against outliers |
| Do two things move together? | **Scatter plot** | Height vs. diameter | Shrink or fade the dots when you have lots of data |
| How does it change over time? | **Line chart** | *(We don't have planting dates, so not today!)* | Watch for seasonal patterns |
| Where is it happening? | **Map** | Cherry tree locations | Compare shares (%), not just counts |


The checks we did in Part 1 aren't just about cleaning. They help us **feel confident in the information our charts communicate**:

| Meet-your-data check | Why it matters for EDA |
|---|---|
| **Missing values** | Missing data can quietly drop out of a chart, or show up as its own bar |
| **Data types** | Categories → bar charts; numbers → histograms and scatter plots |
| **Inconsistencies** | Typos split one group into two bars; duplicates inflate counts |
| **Impossible values & outliers** | One extreme value can stretch an axis and squash the real pattern |

**The takeaway:** good EDA starts with good data. The time you spent meeting your data pays off here.

In [ ]:
import matplotlib.pyplot as plt

# Start here if you skipped Part 1: this loads the clean data we saved
trees = pd.read_csv("data/processed/trees_clean.csv")
trees.shape

## Creating an EDA Chart
Every time you make a chart, walk through three quick steps:
1. **What's my question?**
2. **Which pointer is it?** (one variable at a time? compare & connect?)
3. **What kind of data?** (category or number?) → **which chart?**

### EXAMPLE: Exploring Tree Diameters

1. **Question:** How big are Vancouver's trees? How are trunk diameters spread out?
2. **Pointer:** *One variable at a time*: we're looking at a single column, `diameter_cm`.
3. **Data type:** diameter is a **number** → so we use a **histogram**.

In [ ]:
ax = trees["diameter_cm"].plot(kind="hist", bins=50, color="mediumseagreen", figsize=(8, 5))
ax.set_title("How wide are Vancouver's trees?")
ax.set_xlabel("Trunk diameter (cm)")
ax.set_ylabel("Number of trees")
plt.tight_layout()
plt.show()

 Most trees are on the left: **about 78% are under 50 cm wide**, and the typical (median) tree is about 25 cm. Then there's a long tail of a few very big trees stretching to the right (that's why the x-axis goes all the way out to almost 500 cm).

That shape is called **right-skewed**. It suggests the city has lots of young or small street trees and only a few giants. It's also why we often use the **median** rather than the mean in this dataset: those few giants pull the mean up.

Notice every chart has a **title** and **axis labels**. Try to do the same in yours!

**TRY IT YOURSELF (Easy):** **Which 5 neighbourhoods have the fewest trees?** Before coding: which pointer is this, and which chart fits?

*Hint: Neighbourhood is a **category**, and we're counting. Filter out `"Unknown"` first, then use `value_counts().nsmallest(5)` and `.plot(kind="barh")`.*

In [ ]:
# your code here

<details>
<summary>Show solution</summary>

```python
known = trees[trees["neighbourhood"] != "Unknown"]
fewest = known["neighbourhood"].value_counts().nsmallest(5).sort_values(ascending=False)

ax = fewest.plot(kind="barh", color="mediumseagreen")
ax.set_title("5 neighbourhoods with the fewest trees")
ax.set_xlabel("Number of trees")
ax.set_ylabel("")
plt.tight_layout()
plt.show()
```

**Pointer:** one variable at a time (what's rare?). **Chart:** bar chart, because neighbourhood is a **category**. Stanley Park, Strathcona, South Cambie, the West End and Fairview have the fewest trees. Many of these are smaller or denser areas, so a *count* doesn't tell the whole story!

</details>

**TRY IT YOURSELF (Medium):** **Do the 5 most common tree families (genus) differ in height?** Which pointer is this, and which chart fits?

*Hint: We're comparing a **number** (height) across **groups** (genus). Keep only the top 5 genera with `.isin()`, then try `.boxplot(column="height_m", by="genus")`.*

In [ ]:
# your code here

<details>
<summary> Show solution</summary>

```python
top5 = trees["genus"].value_counts().head(5).index
subset = trees[trees["genus"].isin(top5)]

subset.boxplot(column="height_m", by="genus", figsize=(8, 5))
plt.suptitle("")   # removes an automatic extra title
plt.title("Tree height by genus (top 5)")
plt.xlabel("Genus")
plt.ylabel("Height (m)")
plt.tight_layout()
plt.show()
```

**Pointer:** compare & connect (how do groups differ?). **Chart:** box plot, which compares a **number across groups**. Oaks (QUERCUS) and lindens (TILIA) tend to be the tallest, with a median of about 14 m, while maples (ACER) and cherries/plums (PRUNUS) are the shortest at about 8 m. The dots are outliers. Why does the TILIA box look squashed flat? So many lindens are recorded at exactly 13.7 m that the box collapses: another clue that heights were recorded in ranges!

</details>

**TRY IT YOURSELF (Hard):** **Do taller trees tend to be wider?** Which pointer is this, and which chart fits?

*Hint: Two **numbers** → scatter plot. With ~180,000 trees the dots pile up, so plot a random sample with `trees.sample(2000, random_state=1)` and make the dots see-through with `alpha=0.3`.*

In [ ]:
# your code here

<details>
<summary>Show solution</summary>

```python
sample = trees.sample(2000, random_state=1)

ax = sample.plot(kind="scatter", x="diameter_cm", y="height_m",
                 alpha=0.3, s=10, color="mediumseagreen", figsize=(8, 5))
ax.set_title("Height vs. diameter (2,000 random trees)")
ax.set_xlabel("Trunk diameter (cm)")
ax.set_ylabel("Height (m)")
plt.tight_layout()
plt.show()
```

**Pointer:** compare & connect (do two things move together?). **Chart:** scatter plot, for **two numbers**. The dots trend upward: taller trees tend to be wider. Did you notice the dots form **horizontal stripes**? That's because heights were recorded in ranges, not measured exactly. And that lonely dot up at 70 m? It's our pyramidal hornbeam from Part 1! Surprises worth following.

</details>

---

# Part 3: Data Modelling

EDA helped us **describe** our data. Modelling lets us go further: using patterns in the data to **predict** something, **explain** what matters, or **find groups**.

### What kind of model do I need?
The first question to ask: **do I have a specific *target*, something I want to predict?**

| | Type | Question it answers | Example |
|---|---|---|---|
| **Yes → supervised** | **Classification** | Which category? | Is this tree a cherry? |
| | **Regression** | How much? | How wide is this tree? |
| | **Forecasting** | What happens next over time? | How many visitors next spring? |
| **No → unsupervised** | **Clustering** | What natural groups exist? | Neighbourhoods with similar tree mixes |
| | **Anomaly detection** | What's unusual? | A 70 m-tall street tree |

### A few common models (a quick reference)
There are *lots* of models, from simple to complex. **You don't need to know them all**: simple models are a great place to start, since they're easier to explain and often work surprisingly well.

**Start simple:** begin with a baseline and one or two simple models. Only reach for something more complex if it clearly helps, and if you can still explain it to the judges!

### The modelling workflow
1. **Choose your target & features:** what are you predicting, and with what clues?
2. **Split the data:** train on some, test on data the model hasn't seen
3. **Start with a baseline:** what's the simplest possible guess?
4. **Train & compare models:** try a few; does one do better?
5. **Evaluate:** is it good *for your client's question*?

In [ ]:
import time
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, recall_score, mean_absolute_error

In [ ]:
# Make sure our cherry column exists (in case you started Part 3 fresh)
trees["is_cherry"] = (trees["common_name"].str.contains("CHERRY")
                      & ~trees["common_name"].str.contains("PLUM"))

---

## EXAMPLE: Classification - Is this tree a cherry?

### Step 1: Choose the target & features
- **Target** (what we predict): `is_cherry`, True or False
- **Features** (the clues): `height_m`, `diameter_cm` and `neighbourhood`

Models only understand numbers, so we turn `neighbourhood` into many True/False columns, one per neighbourhood. This is called **one-hot encoding**, and `pd.get_dummies()` does it for us.

In [ ]:
# 1. Remove trees with an unknown neighbourhood
model_data = trees[trees["neighbourhood"] != "___"]

# 2. Target: what we want to predict
y = model_data["___"]

# 3. Features: the clues we'll use (one-hot encode the neighbourhood column)
X = pd.___(model_data[["___", "___", "neighbourhood"]],
           columns=["___"])

# 4. Check the result
print("Rows:", X.shape[0], " Feature columns:", X.shape[1])
X.iloc[:3, :6]

### Step 2: Split the data
We keep **20% of trees hidden** as a test set. The model learns from the other 80%, then we check how well it does on trees it has never seen, like studying with practice questions and then writing the real exam.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    ..., ..., test_size=..., random_state=42)

print("Training trees:", len(X_train))
print("Test trees:    ", len(X_test))
print("Cherries in the test set:", y_test.sum())

### Step 3: Start with a baseline
**DUMMY MODEL** - the simplest possible model: **always guess the most common answer**, which is *not a cherry*.

We'll check two scores for every model (plus how long it takes to train):
- **Accuracy:** what % of all trees did we label correctly?
- **Recall:** of the real cherries, what % did we find?

This small helper function trains a model and adds its scores to a results table, so we don't repeat the same code for every model.

In [ ]:
results = []   # tip: re-run this cell to clear the table if you get duplicate rows

def evaluate(name, model):
    start = time.time()
    model.fit(X_train, y_train)            # learn from the training data
    seconds = time.time() - start
    predictions = model.predict(X_test)    # predict on unseen trees
    results.append({
        "model": name,
        "accuracy_%": round(accuracy_score(y_test, predictions) * 100, 1),
        "recall_%": round(recall_score(y_test, predictions) * 100, 1),
        "train_seconds": round(seconds, 2),
    })
    return pd.DataFrame(results)

evaluate("Baseline: always 'not a cherry'", DummyClassifier(strategy="most_frequent"))

**About 90% accurate... but 0% recall!** Because only ~10% of trees are cherries, a model that never says "cherry" is still right 90% of the time.

**Lesson:** accuracy can be very misleading when the thing you're looking for is rare. Always compare against a baseline!

### Step 4: Train & compare models
Let's try two common beginner-friendly models:
- **Logistic regression:** draws a boundary between "cherry" and "not cherry" (despite the name, it's for classification!)
- **Decision tree:** learns a series of yes/no questions, like a flowchart. `max_depth=5` limits it to 5 questions deep so it doesn't memorize the training data.

In [ ]:
evaluate("Logistic regression", LogisticRegression(max_iter=1000))
evaluate("Decision tree", DecisionTreeClassifier(max_depth=5, random_state=42))

Both models beat the baseline's accuracy, but only by a little, and they still miss **most** of the cherries.

### Step 5: Evaluate: what does our *client* care about?
If the City wants to find blossom spots, **missing cherries is worse than a few false alarms**. We can tell the model that cherries matter more with `class_weight="balanced"`.

In [ ]:
tree_balanced = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")
evaluate("Decision tree (cherries matter more)", tree_balanced)

**The trade-off:** accuracy drops to about 84%, but now we catch about **62% of the cherries**, up from 24%. Which model is "best" depends on what your client cares about. That's a decision worth explaining to the judges!

 **A model is only as good as its features.** With only size and location, there's a limit to how well we can spot cherries. If we had flower colour or bloom time, it would do much better.

**Lesson:** more complex doesn't always mean more accurate. A slightly better score isn't always worth losing the ability to explain your model.

### 3. Match the model to your question
There's no single "best" model; it depends on what your client needs. For example:

| If your client wants to... | A good choice might be... | Because... |
|---|---|---|
| **Understand *why*** (explain to the City) | A small decision tree or logistic regression | Easy to read and explain |
| **Catch as many cherries as possible** | A model with `class_weight="balanced"` | It focuses on the rare group |
| **Squeeze out the best score** (e.g. a leaderboard) | Random forest or gradient boosting | Often a bit more accurate, if you have time |

For our client (the City promoting blossom spots), we'd probably pick the **simple balanced decision tree**: it catches most cherries *and* we can explain how it works.